# Entrenar "oye matix" (openWakeWord, espanol)

Genera un modelo de wake word para **"oye matix"** y exporta un `oye_matix.onnx` compatible con el pipeline de Matix (cadena mel -> embedding -> clasificador; entrada `[1,16,96]`, salida `[1,1]`).

**ANTES DE CORRER:** Entorno de ejecucion -> Cambiar tipo de entorno -> **GPU (T4)**.

VIa: piper-sample-generator NO tiene modelo en espanol, asi que sintetizamos los positivos con el CLI estandar de Piper sobre 8 voces en espanol, y usamos openWakeWord para negativos + aumentacion + entrenamiento + export. ~20-60 min en T4 (domina la descarga de datos).

In [ ]:
!nvidia-smi

## 1. Instalar openWakeWord + Piper + extras

In [ ]:
!git clone https://github.com/dscripka/openWakeWord
%pip install -q -e ./openWakeWord
%pip install -q piper-tts
%pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 \
  speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 \
  acoustics==0.2.6 'datasets[audio]==2.14.6'
# (tensorflow/onnx_tf solo si quieres .tflite; para ONNX no hacen falta y dan conflictos)

## 2. Descargar 8 voces de Piper en espanol

In [ ]:
import os, urllib.request
BASE = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/es'
voices = [('es_AR','daniela','high'), ('es_ES','carlfm','x_low'), ('es_ES','davefx','medium'),
          ('es_ES','sharvard','medium'), ('es_ES','mls_9972','low'), ('es_ES','mls_10246','low'),
          ('es_MX','ald','medium'), ('es_MX','claude','high')]
os.makedirs('voices', exist_ok=True)
for loc, name, q in voices:
    stem = f'{loc}-{name}-{q}'
    for ext in ('.onnx', '.onnx.json'):
        try:
            urllib.request.urlretrieve(f'{BASE}/{loc}/{name}/{q}/{stem}{ext}', f'voices/{stem}{ext}')
            print('ok', stem + ext)
        except Exception as e:
            print('FALLO', stem + ext, e)

## 3. Sintetizar "oye matix" en todas las voces (con variacion) y remuestrear a 16 kHz mono

openWakeWord EXIGE 16 kHz mono; las voces medium/high de Piper salen a 22050 Hz, por eso remuestreamos con ffmpeg. Si los flags `--length_scale`/`--noise_scale` de tu version de piper-tts difieren, ajusta la linea del subprocess (es la parte mas fragil).

In [ ]:
import os, subprocess, uuid, itertools, glob
ROOT = '/content/my_custom_model/oye_matix'
os.makedirs(ROOT + '/positive_train', exist_ok=True)
os.makedirs(ROOT + '/positive_test', exist_ok=True)
PHRASE = 'oye matix'
length_scales = [0.8, 1.0, 1.2]; noise = [0.5, 0.667, 0.8]
def synth(outdir, reps):
    vs = [v for v in os.listdir('voices') if v.endswith('.onnx')]
    for v in vs:
        for ls, ns in itertools.product(length_scales, noise):
            for _ in range(reps):
                tmp = '/content/_t.wav'; fn = os.path.join(outdir, uuid.uuid4().hex + '.wav')
                try:
                    subprocess.run(['piper', '-m', f'voices/{v}', '-f', tmp,
                                    '--length_scale', str(ls), '--noise_scale', str(ns)],
                                   input=PHRASE.encode(), check=True,
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', fn],
                                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                except Exception as e:
                    print('synth fallo', v, e); return
synth(ROOT + '/positive_train', reps=40)  # ~8*9*40 = 2880 clips
synth(ROOT + '/positive_test', reps=6)
print('train:', len(glob.glob(ROOT + '/positive_train/*.wav')),
      'test:', len(glob.glob(ROOT + '/positive_test/*.wav')))

## 4. Descargar negativos + RIR + features precomputadas (~varios GB)

In [ ]:
import os
from huggingface_hub import hf_hub_download, snapshot_download
os.makedirs('mit_rirs', exist_ok=True)
snapshot_download('davidscripka/MIT_environmental_impulse_responses', repo_type='dataset', local_dir='mit_rirs')
hf_hub_download('davidscripka/openwakeword_features', 'openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
               repo_type='dataset', local_dir='/content')
hf_hub_download('davidscripka/openwakeword_features', 'validation_set_features.npy',
               repo_type='dataset', local_dir='/content')
print('listo')

## 5. Config YAML

In [ ]:
cfg = '''
target_phrase: ["oye matix"]
model_name: "oye_matix"
output_dir: "/content/my_custom_model"
piper_sample_generator_path: "./piper-sample-generator"
n_samples: 2880
n_samples_val: 432
tts_batch_size: 50
rir_paths: ["/content/mit_rirs"]
background_paths: []
background_paths_duplication_rate: []
custom_negative_phrases: []
augmentation_rounds: 1
augmentation_batch_size: 16
feature_data_files:
  ACAV100M_sample: "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
false_positive_validation_data_path: "/content/validation_set_features.npy"
model_type: "dnn"
layer_size: 32
steps: 20000
max_negative_weight: 1500
target_accuracy: 0.6
target_recall: 0.25
target_false_positives_per_hour: 0.2
'''
open('/content/oye_matix.yaml', 'w').write(cfg)
print(cfg)

## 6. Aumentar + features (NO `--generate_clips`: ya tenemos los positivos en espanol)

In [ ]:
!cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --augment_clips

## 7. Entrenar el clasificador

In [ ]:
!cd openWakeWord && python openwakeword/train.py --training_config /content/oye_matix.yaml --train_model

## 8. Verificar interfaz ONNX (entrada [1,16,96] f32, salida [1,1])

In [ ]:
import onnx, glob
p = glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0]
m = onnx.load(p)
print('archivo:', p)
print('inputs :', [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input])
print('outputs:', [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output])

## 9. Descargar el modelo

In [ ]:
from google.colab import files
import glob
files.download(glob.glob('/content/my_custom_model/**/oye_matix.onnx', recursive=True)[0])

## 10. Integrarlo en Matix

1. Pon `oye_matix.onnx` en `app/assets/models/wakeword/`.
2. En `app/lib/features/wakeword/data/wakeword_modelo.dart`: `archivo = 'oye_matix.onnx'`, `frase = 'oye matix'`.
3. `flutter analyze && flutter test`, build e instala. La cadena mel/embedding no cambia y el service nativo recibe el nombre por el canal.

Si detecta poco: sube `reps`, agrega 20-50 grabaciones reales tuyas en `positive_train`, o baja el umbral con el slider de Ajustes.